# 冻结目标局部梯度，不训练
只测量共享编码器在成熟训练域中的损失分量梯度，不更新权重，不重放优化轨迹，不用锚点未来收益调损失。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import torch,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.model.minute_pit_allocator import _scaled
from src.workflow.pit_allocation_retrain import _mature_training_labels,PRIMARY_LABELS
from src.workflow.pit_constituent_auxiliary_training_allocation import PitAuxiliaryAllocationNet,AuxiliaryTrainingCandidate,_project_legal_weights,_allocation_loss
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';torch.set_num_threads(1);card=root/'research/frozen_objective_gradient_20261014';plan=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013';qlib.init(provider_uri=str(provider),region='us')
s,audit=_source(root,plan['source'],['inputs/daily_frame','inputs/features','inputs/common','inputs/auxiliary','inputs/frozen_parent_models','inputs/plan']);state=s['inputs/frozen_parent_models']['outer_refit'];spec=s['inputs/plan'];candidate=AuxiliaryTrainingCandidate(**state['candidate'])
for field in ['absolute_drawdown_penalty','conditional_drawdown_penalty','rolling_5d_penalty','cash_target_penalty','trade_event_penalty']:assert getattr(candidate,field)==0
frame,maturity=_mature_training_labels(s['inputs/daily_frame'],plan['cutoff']);train=frame.loc[spec['train_start']:plan['cutoff']].dropna(subset=PRIMARY_LABELS)
features=s['inputs/features'].reindex(train.index);common=s['inputs/common'].reindex(train.index)
aux=s['inputs/auxiliary']['outer'].dropna(subset=[*state['common_features'],'synthetic_2x_crash_label','synthetic_2x_upside_label','training_weight']);aux=aux.loc[aux['$pit_available'].gt(.5)]
assert len(aux)>0 and len(train)>0
net=PitAuxiliaryAllocationNet(primary_feature_count=len(state['primary_features']),common_feature_count=len(state['common_features']),hidden_units=spec['hidden_units'],tail_hidden_units=spec['tail_hidden_units']);net.load_state_dict({k:torch.tensor(v) for k,v in state['model_state'].items()});net.eval()
def identity():return hashlib.sha256(b''.join(v.detach().numpy().tobytes() for v in net.state_dict().values())).hexdigest()
before=identity();logits,tail,alpha=net(_scaled(features,state['primary_features'],state['primary_scaler']),_scaled(common,state['common_features'],state['common_scaler']))
w=_project_legal_weights(logits,maximum_soxl_weight=spec['maximum_soxl_weight'],minimum_cash_weight=spec['minimum_cash_weight'])
def tensor(v):return torch.tensor(np.asarray(v,dtype=np.float32))
allocation,turnover,annual,_=_allocation_loss(weights=w,returns=tensor(train[['soxl_forward_open_return','soxx_forward_open_return']]),years=torch.tensor(train.index.year.to_numpy(),dtype=torch.int64),cash_target=tensor(train.tail_cash_target_5d),tail_event=tensor(train.tail_event_5d),candidate=candidate,drawdown_target_pct=-30.,worst_5d_target_pct=-15.,minimum_cash_weight=spec['minimum_cash_weight'],cost_bps=spec['training_cost_bps'],turnover_weight=spec['turnover_loss_weight'])
_,aux_tail,aux_alpha=net(_scaled(features.iloc[:1],state['primary_features'],state['primary_scaler']).expand(len(aux),-1),_scaled(aux,state['common_features'],state['common_scaler']))
aux_weight=tensor(aux.training_weight);aux_weight=aux_weight/aux_weight.mean();bce=torch.nn.functional.binary_cross_entropy_with_logits
terms={'relative_return':-annual.mean(),'annual_dispersion':.5*annual.std(unbiased=False),'turnover_preference':spec['turnover_loss_weight']*turnover.mean(),'primary_tail':spec['primary_tail_loss_weight']*bce(tail,tensor(train.tail_event_5d)),'primary_upside':spec['primary_tail_loss_weight']*bce(alpha,tensor(train.relative_alpha_5d.gt(0))),'auxiliary_tail':candidate.auxiliary_weight*(bce(aux_tail,tensor(aux.synthetic_2x_crash_label),reduction='none')*aux_weight).mean(),'auxiliary_upside':candidate.auxiliary_weight*(bce(aux_alpha,tensor(aux.synthetic_2x_upside_label),reduction='none')*aux_weight).mean()}
torch.testing.assert_close(sum(terms[k] for k in ['relative_return','annual_dispersion','turnover_preference']),allocation)
parameters=dict(net.named_parameters());blocks={'shared_encoder':[k for k in parameters if k.startswith('tail_encoder')],'allocation_head':[k for k in parameters if k.startswith('allocation_head')],'classification_heads':[k for k in parameters if k.startswith(('tail_head','alpha_head'))]}
gradients={};rows=[]
for name,term in terms.items():
 grad=torch.autograd.grad(term,list(parameters.values()),retain_graph=True,allow_unused=True)
 gradients[name]={k:torch.zeros_like(p) if g is None else g.detach() for (k,p),g in zip(parameters.items(),grad)}
 row={'term':name,'weighted_loss':float(term.detach())}
 for block,names in blocks.items():row[block+'_gradient_norm']=float(torch.linalg.vector_norm(torch.cat([gradients[name][n].flatten() for n in names])))
 rows.append(row)
aggregates={'allocation':['relative_return','annual_dispersion','turnover_preference'],'primary_classification':['primary_tail','primary_upside'],'auxiliary_classification':['auxiliary_tail','auxiliary_upside']}
vectors={name:torch.cat([sum(gradients[t][p] for t in terms_in_group).flatten() for p in blocks['shared_encoder']]) for name,terms_in_group in aggregates.items()}
cosines=[]
for left,right in [('allocation','primary_classification'),('allocation','auxiliary_classification'),('primary_classification','auxiliary_classification')]:
 a,b=vectors[left],vectors[right];cosines.append({'left':left,'right':right,'cosine':float(torch.dot(a,b)/(a.norm()*b.norm())),'left_norm':float(a.norm()),'right_norm':float(b.norm())})
assert before==identity() and all(p.grad is None for p in net.parameters())
summary={'experiment':card.name,'decision':'local_objective_gradient_diagnostic_no_optimizer_steps','new_fits':0,'optimizer_steps':0,'new_backtests':0,'objective_achieved':False,'training_rows':len(train),'auxiliary_rows':len(aux),'training_cutoff':plan['cutoff'],'model_weights_unchanged':True,'original_training_trajectory_reconstructed':False,'auxiliary_role':'Full qualifying population weighted expectation, not original random minibatch or original epoch replay','scope':'Local gradients under frozen research inputs; opposing gradients at a compromise solution do not prove negative transfer or a causal reason for high cash. AdamW moments, weight decay and optimization history are not reconstructed.','provider_uri':str(provider),'region':'us','as_of':spec['as_of'],'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/recorder.html','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/gradients','sources/*']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='no_fit_objective_gradient',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,optimizer_steps=0,training_cutoff=plan['cutoff'])
 rec.save_objects(artifact_path='inputs',plan=plan,source_audit=audit,maturity=maturity,candidate=state['candidate'],source_spec=spec)
 rec.save_objects(artifact_path='analysis',terms=pd.DataFrame(rows),cosines=pd.DataFrame(cosines),gradients=gradients,summary=summary)
 rec.save_objects(artifact_path='sources',objective=(root/'src/workflow/pit_constituent_auxiliary_training_allocation.py').read_text())
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');pd.DataFrame(rows).to_csv(card/'loss_gradients.csv',index=False);pd.DataFrame(cosines).to_csv(card/'shared_gradient_alignment.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False))
assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256']
print(pd.DataFrame(rows).to_string(index=False));print(pd.DataFrame(cosines).to_string(index=False))


# 冻结目标诊断：任务近乎正交，不能直接认定负迁移

本轮0拟合、0优化器步、0回测。加载现役f696冻结权重，按固定2022-12-30信息截止核查3014个成熟主样本和20741个辅助样本，测量原损失系数下的局部梯度；未改变任何权重、标准化、目标仓位或生产release。没有用2023-11-03之后的收益决定参数。

配置损失依次含相对SOXX收益、年度均值分散和换手偏好；训练收益另扣10bp成本。当前固定候选的额外回撤、现金监督和交易事件项系数均为0。主分类权重0.02，辅助分类0.05；主现金输出不是校准的现金优势概率，共享编码器同时承担不同分类任务。

共享编码器的局部梯度范数：相对收益0.0005784，年度分散0.0003869，换手0.0002663；三项合成的配置梯度0.0001139。主尾部/上涨分类合成0.0014077，辅助尾部/上涨分类合成0.0022115。主分类与辅助分类分别是配置净梯度约12.36/19.41倍，但相对收益单项尚有0.0005784，净值比受配置内部抵消影响；不以损失值大小或净梯度比断言整个训练由哪个任务决定。

配置与主分类、配置与辅助分类、主分类与辅助分类的共享梯度余弦分别0.02354、0.01702、0.01239，接近正交，不是显著反向。本诊断没有支持“任务互相打架导致高现金”的简单叙事。分类项对allocation_head的直接梯度为0，影响经共享编码器传递；配置项不直接更新分类头。

边界：这是固定权重在固定研究输入上的局部诊断，不是原始训练轨迹重放。辅助项使用全部合格样本的加权期望，不是当时每轮2048随机样本；未重建AdamW动量、权重衰减与梯度裁剪历史。因此不支持立即删辅助任务、调低系数、断开梯度或声称这些改动能增加超额。没有生成任何候选训练结果。

下一步只核对任务定义是否一致：主标的尾部、成分合成杠杆尾部与现金机会成本分别对应什么标签、期限和资产。不能把auxiliary tail分数直接解释成SOXX现金择时置信度。若解释与数据均正确，应承认现有冻结模型的决策有效性不足；不通过继续变换同一批输入来制造新信息。

决定：local_objective_gradient_diagnostic_no_optimizer_steps。原张量哈希前后相同，所有parameter.grad保持None；分项加总与原配置loss一致；原Recorder来源及NAS对象回读通过。稳定净超额目标未达成。

[损失分量与各参数块梯度](loss_gradients.csv) · [共享梯度方向](shared_gradient_alignment.csv)

Recorder `5d72f72fee6c44a082d1cf7ca6e65ad7`。
